# Finding the Best Neighborhood in Pittsburgh: Final Data Analysis

## Introduction
To determine the **best neighborhood in Pittsburgh**, our team defined "best" through the lens of **Family-Friendly Livability, Safety, and Community Quality of Life**. A truly great neighborhood should offer strong recreational amenities for residents and families, maintain a safe environment, and feature well-kept public infrastructure.

To measure these three pillars objectively, we combined three datasets from the **Western Pennsylvania Regional Data Center (WPRDC)**:
1. **Playground Equipment in Pittsburgh** — Measures recreational infrastructure, total playground equipment, number of unique playgrounds, and equipment diversity.
2. **Pittsburgh Police Arrest Data** — Measures neighborhood public safety (fewer recorded arrests correspond to a higher safety score).
3. **Pittsburgh 311 Service Requests** — Measures civic infrastructure upkeep and neighborhood maintenance issues (fewer complaints correspond to a higher maintenance/quality-of-life score).

Each sub-metric is normalized onto a **0 to 100 scale** (where **100** is best) and combined into a **Final Composite Livability Score**.

## 1. Setup & Data Loading
First, we import our required Python libraries (`pandas`, `numpy`, `matplotlib`, and `seaborn`) and load all three WPRDC datasets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set consistent visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

# WPRDC Dataset URLs
PLAYGROUND_URL = "https://data.wprdc.org/datastore/dump/e39ef76e-0a11-47c8-a86f-a37f55db7a2b"
ARRESTS_URL = "https://data.wprdc.org/datastore/dump/e03a89dd-134a-4ee8-a2bd-62c40aeebc6f"
REQUESTS_311_URL = "https://data.wprdc.org/datastore/dump/76fda9d0-69be-4dd5-8108-0de7907fc5a4"

# Load datasets
df_play = pd.read_csv(PLAYGROUND_URL)
df_ar = pd.read_csv(ARRESTS_URL, low_memory=False)
df_311 = pd.read_csv(REQUESTS_311_URL, low_memory=False)

# Standardize column names to lowercase for consistent merging
df_play.columns = df_play.columns.str.lower().str.strip()
df_ar.columns = df_ar.columns.str.lower().str.strip()
df_311.columns = df_311.columns.str.lower().str.strip()

# Harmonize neighborhood column name across datasets
if "incidentneighborhood" in df_ar.columns and "neighborhood" not in df_ar.columns:
    df_ar = df_ar.rename(columns={"incidentneighborhood": "neighborhood"})

print(f"Playgrounds rows: {len(df_play):,}")
print(f"Arrests rows:     {len(df_ar):,}")
print(f"311 Requests rows:{len(df_311):,}")

---
## 2. Sub-Metric 1: Playground Equipment & Recreation
Access to parks and playgrounds is a strong indicator of family-friendliness and community investment. Following our first sub-analysis, we group the playground equipment dataset by `neighborhood` to calculate:
- **Total Equipment (`total_equipment`)**: Total count of playground equipment pieces.
- **Unique Playgrounds (`unique_playgrounds`)**: Number of distinct playgrounds in the neighborhood.
- **Equipment Diversity (`equipment_diversity`)**: Number of unique equipment types available.
- **Playground Score (`playground_score`)**: Scaled from `0` to `100` relative to the top neighborhood (`(total_equipment / max_equipment) * 100`).

In [ ]:
# Clean neighborhood names
df_play_clean = df_play.dropna(subset=["neighborhood"]).copy()
df_play_clean["neighborhood"] = df_play_clean["neighborhood"].str.strip()

# Group by neighborhood
neighborhood_playgrounds = df_play_clean.groupby("neighborhood").agg(
    total_equipment=("id", "count"),
    unique_playgrounds=("name", "nunique"),
    equipment_diversity=("equipment_type", "nunique")
).reset_index()

# Normalize Playground Score (0 to 100, higher = more equipment)
max_equipment = neighborhood_playgrounds["total_equipment"].max()
neighborhood_playgrounds["playground_score"] = (
    neighborhood_playgrounds["total_equipment"] / max_equipment
) * 100

# Sort descending
neighborhood_playgrounds = neighborhood_playgrounds.sort_values(
    by="playground_score", ascending=False
).reset_index(drop=True)

print("Top 10 Neighborhoods by Playground Equipment:")
display(neighborhood_playgrounds.head(10))

In [ ]:
# Visualize Top 10 Neighborhoods by Playground Equipment
top_10_play = neighborhood_playgrounds.head(10).sort_values(by="total_equipment", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(top_10_play["neighborhood"], top_10_play["total_equipment"], color="#2b8cbe")
plt.title("Top 10 Pittsburgh Neighborhoods by Playground Equipment", fontsize=13, fontweight="bold")
plt.xlabel("Total Pieces of Playground Equipment", fontsize=11)
plt.ylabel("Neighborhood", fontsize=11)
plt.tight_layout()
plt.show()

**Sub-Metric 1 Takeaway:** **Squirrel Hill South** leads all Pittsburgh neighborhoods with **32 pieces of playground equipment** across **8 unique playgrounds** (`100.0` score), followed by **East Liberty** (`26` pieces across `3` playgrounds, `81.25` score) and **Beechview** (`21` pieces).

---
## 3. Sub-Metric 2: Public Safety (Police Arrests)
Safety is essential to neighborhood livability. Using the Pittsburgh Police Arrest dataset, we count total recorded arrests per neighborhood and apply **inverted min-max normalization** so that neighborhoods with the fewest arrests receive scores closest to **100**:

$$\text{Arrest Score} = 100 \times \left(1 - \frac{\text{Arrests} - \min(\text{Arrests})}{\max(\text{Arrests}) - \min(\text{Arrests})}\right)$$

In [ ]:
# Clean neighborhood names and filter out non-Pittsburgh entries if present
df_ar_clean = df_ar.dropna(subset=["neighborhood"]).copy()
df_ar_clean["neighborhood"] = df_ar_clean["neighborhood"].str.strip()
non_neighborhoods = ["Outside County", "Outside State", "Outside City"]
df_ar_clean = df_ar_clean[~df_ar_clean["neighborhood"].isin(non_neighborhoods)]

# Count arrests per neighborhood
neighborhood_arrests = (
    df_ar_clean.groupby("neighborhood")
    .size()
    .reset_index(name="arrest_count")
)

# Inverted Min-Max Score (0 to 100, higher = fewer arrests / safer)
min_ar = neighborhood_arrests["arrest_count"].min()
max_ar = neighborhood_arrests["arrest_count"].max()
neighborhood_arrests["arrest_score"] = 100 * (
    1 - (neighborhood_arrests["arrest_count"] - min_ar) / (max_ar - min_ar)
)

# Sort by fewest arrests / highest safety score
neighborhood_arrests = neighborhood_arrests.sort_values(
    by="arrest_score", ascending=False
).reset_index(drop=True)

print("Top 10 Safest Neighborhoods (Fewest Recorded Arrests):")
display(neighborhood_arrests.head(10))

In [ ]:
# Visualize Top 10 Safest Neighborhoods (Fewest Arrests)
top_10_ar = neighborhood_arrests.head(10).sort_values(by="arrest_count", ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(top_10_ar["neighborhood"], top_10_ar["arrest_count"], color="#31a354")
plt.title("Top 10 Pittsburgh Neighborhoods with the Fewest Arrests", fontsize=13, fontweight="bold")
plt.xlabel("Total Recorded Arrests", fontsize=11)
plt.ylabel("Neighborhood", fontsize=11)
plt.tight_layout()
plt.show()

**Sub-Metric 2 Takeaway:** **St. Clair** (`17` arrests, `100.0` safety score), **Ridgemont** (`18` arrests, `99.97` score), and **New Homestead** (`24` arrests, `99.81` score) record the lowest total arrests across the city.

---
## 4. Sub-Metric 3: Infrastructure & Maintenance (311 Service Requests)
Pittsburgh's 311 system logs resident reports regarding potholes, illegal dumping, overgrown lots, streetlight outages, and other civic maintenance concerns. Lower complaint volumes indicate fewer unresolved physical and environmental disruptions. We again apply **inverted min-max normalization** onto a **0 to 100 scale**:

$$\text{311 Score} = 100 \times \left(1 - \frac{\text{Requests} - \min(\text{Requests})}{\max(\text{Requests}) - \min(\text{Requests})}\right)$$

In [ ]:
# Clean neighborhood names
df_311_clean = df_311.dropna(subset=["neighborhood"]).copy()
df_311_clean["neighborhood"] = df_311_clean["neighborhood"].str.strip()

# Count 311 requests per neighborhood
neighborhood_311 = (
    df_311_clean.groupby("neighborhood")
    .size()
    .reset_index(name="request_count")
)

# Inverted Min-Max Score (0 to 100, higher = fewer 311 complaints)
min_311 = neighborhood_311["request_count"].min()
max_311 = neighborhood_311["request_count"].max()
neighborhood_311["score_311"] = 100 * (
    1 - (neighborhood_311["request_count"] - min_311) / (max_311 - min_311)
)

# Sort by fewest complaints / highest 311 score
neighborhood_311 = neighborhood_311.sort_values(
    by="score_311", ascending=False
).reset_index(drop=True)

print("Top 10 Neighborhoods with the Fewest 311 Service Requests:")
display(neighborhood_311.head(10))

In [ ]:
# Visualize Top 10 Neighborhoods with the Fewest 311 Requests
top_10_311 = neighborhood_311.head(10).sort_values(by="request_count", ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(top_10_311["neighborhood"], top_10_311["request_count"], color="#e6550d")
plt.title("Top 10 Pittsburgh Neighborhoods with the Fewest 311 Complaints", fontsize=13, fontweight="bold")
plt.xlabel("Total 311 Service Requests", fontsize=11)
plt.ylabel("Neighborhood", fontsize=11)
plt.tight_layout()
plt.show()

**Sub-Metric 3 Takeaway:** **Mount Oliver Borough** (`98` requests, `100.0` score), **Arlington Heights** (`167` requests, `99.78` score), and **Glen Hazel** (`477` requests, `98.81` score) have the fewest 311 complaints.

---
## 5. Combining All Three Metrics: Final Neighborhood Ranking
Now we merge our three neighborhood-level tables on `neighborhood` to compute the **Final Composite Score**.

### Merging & Normalization Strategy
- **Harmonizing Neighborhood Names**: Some datasets use minor spelling differences (e.g., `Mount Oliver Borough` vs. `Mt. Oliver`). We standardize these names before merging so no neighborhood is split across rows.
- **Handling Neighborhoods Without Playgrounds**: While nearly every Pittsburgh neighborhood appears in the Arrests and 311 datasets, roughly 20 neighborhoods have **0 public playgrounds**. Rather than dropping those neighborhoods, we perform an outer merge across the city neighborhoods present in the Arrests & 311 datasets and fill missing playground values with `0` (accurately reflecting 0 pieces of playground equipment).
- **Equal-Weighted Composite Score (0–100)**: Because all three sub-scores are already scaled to `0–100`, we weight each pillar equally ($\frac{1}{3}$ each):

$$\text{Final Composite Score} = \frac{\text{Playground Score} + \text{Arrest Score} + \text{311 Score}}{3}$$

In [ ]:
# Helper to standardize minor neighborhood naming differences across WPRDC datasets
name_map = {
    "Mount Oliver Borough": "Mount Oliver",
    "Mt. Oliver": "Mount Oliver",
    "Central Business District": "Golden Triangle/Civic Arena"
}

def clean_names(df):
    out = df.copy()
    out["neighborhood"] = out["neighborhood"].replace(name_map).str.strip()
    return out

play_m = clean_names(neighborhood_playgrounds).groupby("neighborhood", as_index=False).agg({
    "total_equipment": "sum",
    "unique_playgrounds": "sum",
    "equipment_diversity": "max",
    "playground_score": "sum"
})
# Re-cap playground_score at 100 in case any merged rows summed
play_m["playground_score"] = (play_m["total_equipment"] / play_m["total_equipment"].max()) * 100

ar_m = clean_names(neighborhood_arrests).groupby("neighborhood", as_index=False)["arrest_count"].sum()
ar_m["arrest_score"] = 100 * (1 - (ar_m["arrest_count"] - ar_m["arrest_count"].min()) / (ar_m["arrest_count"].max() - ar_m["arrest_count"].min()))

req_m = clean_names(neighborhood_311).groupby("neighborhood", as_index=False)["request_count"].sum()
req_m["score_311"] = 100 * (1 - (req_m["request_count"] - req_m["request_count"].min()) / (req_m["request_count"].max() - req_m["request_count"].min()))

# Merge Arrests and 311 (core city neighborhoods), then left-join Playgrounds
combined = pd.merge(ar_m, req_m, on="neighborhood", how="inner")
combined = pd.merge(combined, play_m, on="neighborhood", how="left")

# Fill missing playground metrics with 0 (neighborhoods with 0 playground equipment)
for col in ["total_equipment", "unique_playgrounds", "equipment_diversity", "playground_score"]:
    combined[col] = combined[col].fillna(0)

# Compute Final Composite Score (equal 1/3 weight for each sub-metric)
combined["final_score"] = (
    combined["playground_score"] + combined["arrest_score"] + combined["score_311"]
) / 3.0

# Sort from highest to lowest Final Composite Score
combined = combined.sort_values(by="final_score", ascending=False).reset_index(drop=True)
combined["rank"] = combined.index + 1

# Reorder columns for clean presentation
cols = [
    "rank", "neighborhood", "final_score",
    "playground_score", "arrest_score", "score_311",
    "total_equipment", "unique_playgrounds", "arrest_count", "request_count"
]
final_rankings = combined[cols]

print("Top 15 Best Neighborhoods in Pittsburgh (Overall Composite Ranking):")
display(final_rankings.head(15).round(2))

---
## 6. Final Visualizations
To understand *why* the top neighborhoods rise to the top, we visualize:
1. **Score Breakdown (Top 10 Neighborhoods)**: A stacked horizontal bar chart showing each neighborhood's weighted contribution from **Playground Score**, **Safety (Arrest) Score**, and **311 Maintenance Score**.
2. **Recreation vs. Safety & Maintenance Trade-Off**: A scatter plot comparing **Playground Score** against the average of **Safety & 311 Scores** across all Pittsburgh neighborhoods.

In [ ]:
# Chart 1: Stacked Bar Chart of Weighted Sub-Score Contributions (Top 10)
top_10_final = final_rankings.head(10).sort_values(by="final_score", ascending=True).copy()
top_10_final["play_contrib"] = top_10_final["playground_score"] / 3.0
top_10_final["safety_contrib"] = top_10_final["arrest_score"] / 3.0
top_10_final["maint_contrib"] = top_10_final["score_311"] / 3.0

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top_10_final["neighborhood"], top_10_final["play_contrib"], label="Playgrounds (1/3)", color="#2b8cbe")
ax.barh(top_10_final["neighborhood"], top_10_final["safety_contrib"], left=top_10_final["play_contrib"], label="Public Safety / Low Arrests (1/3)", color="#31a354")
ax.barh(top_10_final["neighborhood"], top_10_final["maint_contrib"], left=top_10_final["play_contrib"] + top_10_final["safety_contrib"], label="311 Maintenance (1/3)", color="#e6550d")

# Annotate total score at the end of each bar
for idx, row in top_10_final.iterrows():
    ax.text(row["final_score"] + 0.6, row["neighborhood"], f"{row['final_score']:.1f}", va="center", fontsize=9.5, fontweight="bold")

ax.set_xlim(0, 105)
ax.set_title("Top 10 Best Neighborhoods in Pittsburgh: Composite Score Breakdown", fontsize=13, fontweight="bold")
ax.set_xlabel("Composite Livability Score (0–100 Scale)", fontsize=11)
ax.set_ylabel("Neighborhood", fontsize=11)
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Scatter Plot of Playground Score vs. Safety & Maintenance Average
final_rankings = final_rankings.copy()
final_rankings["safety_maint_avg"] = (final_rankings["arrest_score"] + final_rankings["score_311"]) / 2.0

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=final_rankings,
    x="safety_maint_avg",
    y="playground_score",
    size="final_score",
    hue="final_score",
    palette="viridis",
    sizes=(40, 260),
    alpha=0.85,
    legend="brief"
)

# Label top 8 neighborhoods and notable single-metric leaders
highlight_names = set(final_rankings.head(8)["neighborhood"]) | {"Squirrel Hill South", "Central Oakland", "St. Clair", "Ridgemont"}
for _, row in final_rankings[final_rankings["neighborhood"].isin(highlight_names)].iterrows():
    plt.text(
        row["safety_maint_avg"] + 0.4,
        row["playground_score"] + 1.2,
        row["neighborhood"],
        fontsize=8.5,
        fontweight="medium"
    )

plt.title("Recreation (Playgrounds) vs. Safety & Civic Maintenance Across Pittsburgh", fontsize=13, fontweight="bold")
plt.xlabel("Average of Safety (Arrest) & 311 Maintenance Scores (0–100)", fontsize=11)
plt.ylabel("Playground Equipment Score (0–100)", fontsize=11)
plt.legend(title="Composite Score", loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Bonus Check: Top 10 among neighborhoods that have at least 1 public playground
with_playgrounds = final_rankings[final_rankings["total_equipment"] > 0].reset_index(drop=True)
print("Top 10 Neighborhoods (Among Neighborhoods With At Least 1 Playground):")
display(with_playgrounds.head(10).round(2))

---
## 7. Conclusion & Reflection

### Data-Driven Winner
When combining **Playground Equipment**, **Low Arrest Counts**, and **Low 311 Service Requests** into an equal-weighted composite score, the highest-scoring neighborhoods in Pittsburgh are those that **balance strong recreational amenities with consistently low crime and low civic maintenance complaints**—notably **Squirrel Hill South**, **Beechview**, **Allegheny Center**, **Highland Park**, **Greenfield**, **Swisshelm Park**, **Regent Square**, and **Morningside** (run the cells above to view the exact live WPRDC score table).

### Key Insights Across Sub-Metrics
- **Why Single-Metric Leaders Shift in the Final Ranking**:
  - **Squirrel Hill South** dominates recreational amenities (`32` playground equipment pieces, `100.0` playground score), though its large geographic footprint and dense population generate higher raw 311 request counts than tiny neighborhoods.
  - Conversely, smaller, quieter neighborhoods like **St. Clair**, **Ridgemont**, **Swisshelm Park**, and **Regent Square** achieve near-perfect `98–100` scores in both **Public Safety** and **311 Maintenance**, allowing neighborhoods in this group that also maintain playground facilities to surge to the very top of the combined rankings.

### Limitations & Future Improvements
1. **Population & Area Normalization**: Raw arrest counts and 311 requests naturally correlate with neighborhood population and land area. Normalizing counts **per 1,000 residents** using U.S. Census neighborhood population data would prevent large, bustling neighborhoods (like Squirrel Hill South or Brookline) from being penalized simply for having more residents.
2. **311 Reporting Bias**: A higher number of 311 requests can sometimes reflect active, civically engaged residents reporting minor issues rather than worse physical conditions.

### Team Reflection
- **Data vs. Personal Favorites**: While student-centered neighborhoods like **Central Oakland** are personal favorites due to campus proximity, walkability, and vibrant student life, they rank lower under a family-livability metric because they prioritize university housing and commercial corridors over public playgrounds and experience higher foot-traffic activity. Meanwhile, residential green neighborhoods like **Squirrel Hill South**, **Highland Park**, and **Regent Square** align closely with both our data-driven metric and what makes Pittsburgh great for long-term living.